# Customer churn: group project portfolio edition

Authors: Md Abdul Wahid Raju, Sazzad Hossain, Shusmita Dutta, Asma Akter.
This notebook adapts the group submission: the local data path and the final comparison table are corrected. Cached outputs were cleared. Reported historical results are documented in ../docs/REVIEW.md; new runs may yield different neural-network scores.


EDA and Task-1


In [ ]:
#imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Place the source CSV in data/ (see data/README.md).
from pathlib import Path
path = Path('data/WA_Fn-UseC_-Telco-Customer-Churn.csv')
df = pd.read_csv(path)
df.shape, df.head()


In [ ]:
#quick_structure_check
df.info()
df.describe(include="all").T

In [ ]:
#find_all_the_unique_values
for col in df.columns:
    print(f"\n{col}")
    print(df[col].unique())

In [ ]:
#fix_all_the_same_unique_values
df_clean = df.copy()
internet_cols = [
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies"
]
for col in internet_cols:
    df_clean[col] = df_clean[col].replace("No internet service", "No")

In [ ]:
df_clean["MultipleLines"] = df_clean["MultipleLines"].replace(
    "No phone service", "No"
)

In [ ]:
df_clean

In [ ]:
#change total charge from string to numeric(#Convert TotalCharges to numeric (blank → NaN))
df_clean["TotalCharges"] = pd.to_numeric(
    df_clean["TotalCharges"], errors="coerce"
)


In [ ]:
#checking the missing values of full dataset
df_clean.isna().sum()


In [ ]:
#check if any Empty strings in object columns
(df_clean == "").sum()


In [ ]:
#Look at those missing rows (very important insight verify)
df_clean[df_clean["TotalCharges"].isna()][
    ["tenure", "MonthlyCharges", "TotalCharges"]
]

In [ ]:
#Imputation: If tenure == 0, set TotalCharges = 0
df_clean["TotalCharges"] = df_clean["TotalCharges"].fillna(0)

df_clean["TotalCharges"].isna().sum()


In [ ]:
#Continue Task 1 (now distributions, outliers, skewness)
#Summary stats for numeric columns
num_cols = ["tenure", "MonthlyCharges", "TotalCharges"]
df_clean[num_cols].describe().T

In [ ]:
#checking the distribution
import seaborn as sns

for col in num_cols:
    sns.histplot(df_clean[col], kde=True)
    plt.title(f"Distribution of {col}")
    plt.show()

In [ ]:
#boxplot_for_outliers
for col in num_cols:
    sns.boxplot(x=df_clean[col])
    plt.title(f"Boxplot of {col}")
    plt.show()

In [ ]:
# Outlier counts using IQR (on cleaned data)
def iqr_outlier_count(s):
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    low, high = q1 - 1.5*iqr, q3 + 1.5*iqr
    count = ((s < low) | (s > high)).sum()
    return count, low, high

for col in ["tenure", "MonthlyCharges", "TotalCharges"]:
    c, low, high = iqr_outlier_count(df_clean[col])
    print(f"{col}: outliers={c}, bounds=({low:.2f}, {high:.2f})")

In [ ]:
#Skewness
df_clean[num_cols].skew()


In [ ]:
#Figuring out some rising quetion below here
#Do customers with higher monthly charges churn more?
#Do long-term customers churn less?
#Does longer tenure reduce churn?

for col in num_cols:
    plt.figure(figsize=(7,5))
    sns.histplot(
        data=df_clean,
        x=col,
        hue="Churn",
        bins=30,
        multiple="layer",      # bars overlap
        stat="count",          # actual counts
        palette={"No": "steelblue", "Yes": "orange"},
        alpha=0.6
    )
    plt.title(f"{col} Distribution by Churn Status")
    plt.xlabel(col)
    plt.ylabel("Count")
    plt.show()

In [ ]:
# Correlation among numeric features (using cleaned data)
num_cols = ["tenure", "MonthlyCharges", "TotalCharges"]

corr = df_clean[num_cols].corr()
print(corr)

import matplotlib.pyplot as plt

plt.figure()
plt.imshow(corr, aspect="auto")
plt.xticks(range(len(num_cols)), num_cols, rotation=45)
plt.yticks(range(len(num_cols)), num_cols)
plt.colorbar()
plt.title("Correlation heatmap (numeric features)")
plt.show()

In [ ]:
# Study churn vs. non-churn population imbalance (using cleaned data)
churn_counts = df_clean["Churn"].value_counts()
churn_percent = df_clean["Churn"].value_counts(normalize=True) * 100

print(churn_counts)
print(churn_percent)

import matplotlib.pyplot as plt

plt.figure()
plt.bar(churn_counts.index, churn_counts.values)
plt.title("Churn vs Non-Churn Distribution")
plt.xlabel("Churn")
plt.ylabel("Count")
plt.show()


In [ ]:
#save “clean base” dataframe
df_clean.to_csv("telcocust_clean.csv", index=False)


Shallow model and Step-2


In [ ]:
df1 = pd.read_csv("telcocust_clean.csv")
df1.head()

In [ ]:
#drop_the_customer_id
df1 = df1.drop(columns=["customerID"])
df1

In [ ]:
#define X and y and manually put 0 and 1 in target varibale
X = df1.drop(columns=["Churn"])
y = df1["Churn"].map({"No": 0, "Yes": 1})  # 0 = No churn, 1 = churn

y.value_counts(normalize=True)

In [ ]:
#Train/Test split (STRATIFIED) + preprocessing
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [ ]:
#Cell 4 :column groups
num_features = ["tenure", "MonthlyCharges", "TotalCharges"]
cat_features = [c for c in X.columns if c not in num_features]

In [ ]:
# Model 1: Logistic Regression (ensemble-compatible: keep TotalCharges)
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, GridSearchCV

# Use the same feature set as RF/GB (do NOT drop TotalCharges)
X_train_lr = X_train.copy()
X_test_lr  = X_test.copy()

num_lr = ["tenure", "MonthlyCharges", "TotalCharges"]
cat_lr = [c for c in X_train_lr.columns if c not in num_lr]

preprocess_lr = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_lr),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_lr)
    ]
)

lr = LogisticRegression(max_iter=2000, class_weight="balanced", solver="liblinear")

pipe_lr = Pipeline(steps=[
    ("prep", preprocess_lr),
    ("model", lr)
])

param_grid_lr = {
    "model__C": [0.01, 0.1, 1, 10]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_lr = GridSearchCV(
    pipe_lr, param_grid_lr, cv=cv, scoring="f1", n_jobs=-1
)

grid_lr.fit(X_train_lr, y_train)

grid_lr.best_params_, grid_lr.best_score_

In [ ]:
#Cell — evaluate logistic regression
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

def evaluate_model(model, X_te, y_te, name="Model"):
    y_pred = model.predict(X_te)
    acc = accuracy_score(y_te, y_pred)
    f1 = f1_score(y_te, y_pred)
    cm = confusion_matrix(y_te, y_pred)
    print(f"=== {name} ===")
    print("Accuracy:", round(acc, 4))
    print("F1-score:", round(f1, 4))
    print("Confusion Matrix:\n", cm)
    print("\nClassification report:\n", classification_report(y_te, y_pred, digits=4))
    return {"model": name, "accuracy": acc, "f1": f1}

best_lr = grid_lr.best_estimator_
res_lr = evaluate_model(best_lr, X_test_lr, y_test, "Logistic Regression (tuned)")

In [ ]:
#Random Forest pipeline + tuning
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    random_state=42,
    class_weight="balanced"
)

pipe_rf = Pipeline(steps=[
    ("prep", ColumnTransformer(
        transformers=[
            ("num", "passthrough", num_features),
            ("cat", OneHotEncoder(handle_unknown="ignore"), cat_features)
        ]
    )),
    ("model", rf)
])

param_grid_rf = {
    "model__n_estimators": [200, 400],
    "model__max_depth": [None, 10, 20],
    "model__min_samples_split": [2, 10],
    "model__min_samples_leaf": [1, 5]
}

grid_rf = GridSearchCV(
    pipe_rf,
    param_grid_rf,
    cv=cv,
    scoring="f1",
    n_jobs=-1
)

grid_rf.fit(X_train, y_train)

grid_rf.best_params_, grid_rf.best_score_


In [ ]:
#Cell — evaluate Random Forest
best_rf = grid_rf.best_estimator_
res_rf = evaluate_model(best_rf, X_test, y_test, "Random Forest (tuned)")

In [ ]:
#Cell — Gradient Boosting tuning
from sklearn.ensemble import GradientBoostingClassifier

gb = GradientBoostingClassifier(random_state=42)

pipe_gb = Pipeline(steps=[
    ("prep", ColumnTransformer(
        transformers=[
            ("num", "passthrough", num_features),
            ("cat", OneHotEncoder(handle_unknown="ignore"), cat_features)
        ]
    )),
    ("model", gb)
])

param_grid_gb = {
    "model__n_estimators": [100, 200],
    "model__learning_rate": [0.05, 0.1],
    "model__max_depth": [3, 5]
}

grid_gb = GridSearchCV(
    pipe_gb,
    param_grid_gb,
    cv=cv,
    scoring="f1",
    n_jobs=-1
)

grid_gb.fit(X_train, y_train)

grid_gb.best_params_, grid_gb.best_score_

In [ ]:
#Cell — evaluate Gradient Boosting
best_gb = grid_gb.best_estimator_
res_gb = evaluate_model(best_gb, X_test, y_test, "Gradient Boosting (tuned)")


In [ ]:
#model comparison table:
results = pd.DataFrame([res_lr, res_rf, res_gb])
results

In [ ]:
#Voting Ensemble(Soft)
best_lr = grid_lr.best_estimator_
best_rf = grid_rf.best_estimator_
best_gb = grid_gb.best_estimator_


In [ ]:
#create the voting classifer
from sklearn.ensemble import VotingClassifier

voting = VotingClassifier(
    estimators=[
        ("lr", best_lr),
        ("rf", best_rf),
        ("gb", best_gb)
    ],
    voting="soft"  # use probabilities
)

In [ ]:
#train voting
voting.fit(X_train, y_train)

In [ ]:
res_voting = evaluate_model(
    voting,
    X_test,
    y_test,
    "Voting Ensemble (soft)"
)


NN and Task-3


In [ ]:
# Load cleaned dataset
df2 = pd.read_csv("telcocust_clean.csv")
df2 = df2.drop(columns=["customerID"])

# Quick look
print(df2.shape)
df2.head()

In [ ]:
# Check churn distribution
print(df2["Churn"].value_counts())

# Convert to numeric if needed
if df2["Churn"].dtype == "object":
    df2["Churn"] = df2["Churn"].map({"No": 0, "Yes": 1})

print(df2["Churn"].value_counts())
print(df2["Churn"].dtype)


In [ ]:
#Separate features (X) and target (y)
X = df2.drop(columns=["Churn"])
y = df2["Churn"].astype(int)

print("X shape:", X.shape)
print("y shape:", y.shape)

In [ ]:
# Convert categorical columns to dummy variables
X_encoded = pd.get_dummies(X, drop_first=True)

print("Before encoding:", X.shape)
print("After encoding:", X_encoded.shape)

In [ ]:
#Train / Validation / Test split (with stratification)
from sklearn.model_selection import train_test_split

# 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42, stratify=y
)

# From train: 80% train, 20% val  -> overall: 64% train, 16% val, 20% test
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42, stratify=y_train
)

print("Train:", X_train.shape, y_train.shape)
print("Val:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

In [ ]:
#Standardize features (VERY important for neural networks)
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_s = scaler.fit_transform(X_train)  # fit only on train
X_val_s   = scaler.transform(X_val)
X_test_s  = scaler.transform(X_test)

print(X_train_s.shape, X_val_s.shape, X_test_s.shape)

In [ ]:
#Import TensorFlow / Keras
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers

In [ ]:
#Define a SIMPLE FFNN model (baseline)
model = keras.Sequential([
    layers.Input(shape=(X_train_s.shape[1],)),  # number of input features = 23

    layers.Dense(64, activation="relu"),

    layers.Dense(32, activation="relu"),

    layers.Dense(1, activation="sigmoid")  # binary classification
])


In [ ]:
#Compile the model (optimizer + loss)
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)


In [ ]:
#model summary
model.summary()


In [ ]:
#Add Regularization (Dropout + L2 + BatchNorm)
from tensorflow import keras
from tensorflow.keras import layers, regularizers

# Regularized FFNN (better than baseline)
reg_model = keras.Sequential([
    layers.Input(shape=(X_train_s.shape[1],)),   # 23 features

    layers.Dense(64, activation="relu",
                 kernel_regularizer=regularizers.l2(1e-4)),
    layers.BatchNormalization(),
    layers.Dropout(0.3),

    layers.Dense(32, activation="relu",
                 kernel_regularizer=regularizers.l2(1e-4)),
    layers.BatchNormalization(),
    layers.Dropout(0.3),

    layers.Dense(1, activation="sigmoid")
])

reg_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

reg_model.summary()


In [ ]:
#Early Stopping callback
early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)


In [ ]:
#Train the model
history = reg_model.fit(
    X_train_s, y_train,
    validation_data=(X_val_s, y_val),
    epochs=200,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)


In [ ]:
#build model with configurable optimizer/LR/activation (+ regularization)
from tensorflow import keras
from tensorflow.keras import layers, regularizers
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def build_ffnn(input_dim, activation="relu", l2=1e-4, dropout=0.3, use_bn=True,
               optimizer_name="adam", lr=1e-3):

    model = keras.Sequential()
    model.add(layers.Input(shape=(input_dim,)))

    # Block 1
    model.add(layers.Dense(64, activation=activation,
                           kernel_regularizer=regularizers.l2(l2)))
    if use_bn:
        model.add(layers.BatchNormalization())
    model.add(layers.Dropout(dropout))

    # Block 2
    model.add(layers.Dense(32, activation=activation,
                           kernel_regularizer=regularizers.l2(l2)))
    if use_bn:
        model.add(layers.BatchNormalization())
    model.add(layers.Dropout(dropout))

    # Output
    model.add(layers.Dense(1, activation="sigmoid"))

    # Optimizer choice
    if optimizer_name.lower() == "adam":
        opt = keras.optimizers.Adam(learning_rate=lr)
    elif optimizer_name.lower() == "sgd":
        opt = keras.optimizers.SGD(learning_rate=lr, momentum=0.9)
    elif optimizer_name.lower() == "rmsprop":
        opt = keras.optimizers.RMSprop(learning_rate=lr)
    else:
        raise ValueError("optimizer_name must be 'adam', 'sgd', or 'rmsprop'")

    model.compile(optimizer=opt, loss="binary_crossentropy", metrics=["accuracy"])
    return model


In [ ]:
#Early stopping
early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)


In [ ]:
experiments = [
    {"name": "Adam lr=1e-3 ReLU", "optimizer": "adam", "lr": 1e-3, "activation": "relu"},
    {"name": "Adam lr=1e-4 ReLU", "optimizer": "adam", "lr": 1e-4, "activation": "relu"},
    {"name": "SGD  lr=1e-2 ReLU", "optimizer": "sgd",  "lr": 1e-2, "activation": "relu"},
    {"name": "Adam lr=1e-3 Tanh", "optimizer": "adam", "lr": 1e-3, "activation": "tanh"},
]

results = []
histories = {}

for exp in experiments:
    print("\n=== Running:", exp["name"], "===")

    model = build_ffnn(
        input_dim=X_train_s.shape[1],
        activation=exp["activation"],
        optimizer_name=exp["optimizer"],
        lr=exp["lr"],
        l2=1e-4,
        dropout=0.3,
        use_bn=True
    )

    h = model.fit(
        X_train_s, y_train,
        validation_data=(X_val_s, y_val),
        epochs=200,
        batch_size=32,
        callbacks=[early_stop],
        verbose=0
    )

    # Predict on validation (or test later). We use VAL for choosing best config.
    y_val_prob = model.predict(X_val_s).ravel()
    y_val_pred = (y_val_prob >= 0.5).astype(int)

    acc = accuracy_score(y_val, y_val_pred)
    f1  = f1_score(y_val, y_val_pred)

    results.append([exp["name"], acc, f1, len(h.history["loss"])])
    histories[exp["name"]] = h

import pandas as pd
results_df = pd.DataFrame(results, columns=["Experiment", "Val_Accuracy", "Val_F1", "Epochs_Trained"])
results_df.sort_values("Val_F1", ascending=False)


In [ ]:
best_name = results_df.sort_values("Val_F1", ascending=False).iloc[0]["Experiment"]
best_hist = histories[best_name]

import matplotlib.pyplot as plt

plt.figure()
plt.plot(best_hist.history["loss"], label="train loss")
plt.plot(best_hist.history["val_loss"], label="val loss")
plt.title(f"Loss Curve - {best_name}")
plt.legend()
plt.show()

plt.figure()
plt.plot(best_hist.history["accuracy"], label="train acc")
plt.plot(best_hist.history["val_accuracy"], label="val acc")
plt.title(f"Accuracy Curve - {best_name}")
plt.legend()
plt.show()


In [ ]:
#Rebuild the BEST model
best_model = build_ffnn(
    input_dim=X_train_s.shape[1],
    activation="tanh",
    optimizer_name="adam",
    lr=1e-3,
    l2=1e-4,
    dropout=0.3,
    use_bn=True
)

best_model.summary()


In [ ]:
#Train best model (with EarlyStopping)
early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

history_best = best_model.fit(
    X_train_s, y_train,
    validation_data=(X_val_s, y_val),
    epochs=200,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)


In [ ]:
#Predict on TEST set
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

# Predict probabilities
y_test_prob = best_model.predict(X_test_s).ravel()

# Convert probabilities to class labels
y_test_pred = (y_test_prob >= 0.5).astype(int)


In [ ]:
#Evaluation metrics
test_acc = accuracy_score(y_test, y_test_pred)
test_f1  = f1_score(y_test, y_test_pred)
cm       = confusion_matrix(y_test, y_test_pred)

print("Test Accuracy:", round(test_acc, 4))
print("Test F1-score:", round(test_f1, 4))
print("Confusion Matrix:\n", cm)

print("\nClassification Report:\n",
      classification_report(y_test, y_test_pred))


Build and train the autoencoder

In [ ]:
#Choose the latent dimension
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt

input_dim = X_train_s.shape[1]   # 23
latent_dim = 8                   # compressed representation size


In [ ]:
#Build the Autoencoder (Encoder + Decoder)
# Encoder
inp = layers.Input(shape=(input_dim,))
x = layers.Dense(32, activation="relu")(inp)
x = layers.Dense(16, activation="relu")(x)
latent = layers.Dense(latent_dim, activation="relu", name="latent")(x)

# Decoder
x = layers.Dense(16, activation="relu")(latent)
x = layers.Dense(32, activation="relu")(x)
out = layers.Dense(input_dim, activation="linear")(x)

# Models
autoencoder = keras.Model(inp, out, name="autoencoder")
encoder = keras.Model(inp, latent, name="encoder")

autoencoder.summary()


In [ ]:
#Compile the Autoencoder
autoencoder.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="mse"
)


In [ ]:
#Train with EarlyStopping
early_ae = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

hist_ae = autoencoder.fit(
    X_train_s, X_train_s,                     # input = output
    validation_data=(X_val_s, X_val_s),
    epochs=200,
    batch_size=32,
    callbacks=[early_ae],
    verbose=1
)


In [ ]:
#Plot reconstruction loss curve
plt.figure()
plt.plot(hist_ae.history["loss"], label="train mse")
plt.plot(hist_ae.history["val_loss"], label="val mse")
plt.title("Autoencoder Reconstruction Loss")
plt.legend()
plt.show()


In [ ]:
#Extract latent embeddings
# Create compressed features using the trained encoder
Z_train = encoder.predict(X_train_s)
Z_val   = encoder.predict(X_val_s)
Z_test  = encoder.predict(X_test_s)

print("Z_train shape:", Z_train.shape)
print("Z_val shape:", Z_val.shape)
print("Z_test shape:", Z_test.shape)


In [ ]:
#Train a classifier on latent features (Logistic Regression)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

lr_latent = LogisticRegression(max_iter=2000, class_weight="balanced")
lr_latent.fit(Z_train, y_train)

y_pred_latent = lr_latent.predict(Z_test)

print("Latent + LR Test Accuracy:", round(accuracy_score(y_test, y_pred_latent), 4))
print("Latent + LR Test F1:", round(f1_score(y_test, y_pred_latent), 4))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_latent))
print("\nClassification Report:\n", classification_report(y_test, y_pred_latent))


In [ ]:
# Derive both neural-model rows from the evaluations above.
comparison = pd.DataFrame({
    "Model": ["Raw FFNN (23 encoded features)",
              "Autoencoder (8 features) + Logistic Regression"],
    "Test Accuracy": [test_acc, accuracy_score(y_test, y_pred_latent)],
    "Test F1-score": [test_f1, f1_score(y_test, y_pred_latent)]
})
comparison


In [ ]:
print(comparison)

In [ ]:
comparison_renamed = comparison.rename(columns={
    "Model": "model",
    "Test Accuracy": "accuracy",
    "Test F1-score": "f1"
})
comparison_renamed

In [ ]:
# Validation experiment records are separate from test results.
# Final cross-family comparison is assembled below after voting evaluation.
print(comparison_renamed)


Task-4 Model Comparison and Interpretability

In [ ]:
# Compare only test-set scores; never mix in validation experiments.
shallow_test_results = [res_lr, res_rf, res_gb, res_voting]
all_model_results = pd.concat(
    [pd.DataFrame(shallow_test_results), comparison_renamed], ignore_index=True
)
print(all_model_results.sort_values(by="f1", ascending=False))


In [ ]:
y_pred_voting = voting.predict(X_test_lr)
print("Predictions obtained for Voting Ensemble model.")

In [ ]:
y_pred_voting = voting.predict(X_test_lr)

# Create a DataFrame for easier analysis of misclassifications, using X_test_lr for column consistency
results_df = X_test_lr.copy()
results_df["Actual_Churn"] = y_test
results_df["Predicted_Churn"] = y_pred_voting

# Identify False Positives: Model predicted churn (1), but actual was no churn (0)
false_positives = results_df[(results_df["Actual_Churn"] == 0) & (results_df["Predicted_Churn"] == 1)]

# Identify False Negatives: Model predicted no churn (0), but actual was churn (1)
false_negatives = results_df[(results_df["Actual_Churn"] == 1) & (results_df["Predicted_Churn"] == 0)]

print(f"Number of False Positives: {len(false_positives)}")
print(f"Number of False Negatives: {len(false_negatives)}\n")

print("--- Examples of False Positives (Predicted Churn, Actual No Churn) ---")
print(false_positives.head())

print("\n--- Examples of False Negatives (Predicted No Churn, Actual Churn) ---")
print(false_negatives.head())

In [ ]:
import shap
print("Shap library imported.")

In [ ]:
import numpy as np

# Extract the trained Random Forest model
# best_rf was already assigned in cell 'lTHUU7rYZMpY' and 'O968OE-LansP'
# best_rf = grid_rf.best_estimator_

# Extract the trained Raw FFNN model
# best_model was already assigned in cell 'Y1Z7jO1Vrysb'
# best_model = build_ffnn(
#     input_dim=X_train_s.shape[1],
#     activation="tanh",
#     optimizer_name="adam",
#     lr=1e-3,
#     l2=1e-4,
#     dropout=0.3,
#     use_bn=True
# )

# Create a background dataset for SHAP (e.g., 100 random samples from X_train_s)
# This is typically used for KernelExplainer or DeepExplainer
background_data_shap = X_train_s[np.random.choice(X_train_s.shape[0], 100, replace=False)]

print(f"Random Forest model: {best_rf}")
print(f"Raw FFNN model: {best_model}")
print(f"SHAP background dataset shape: {background_data_shap.shape}")

In [ ]:
X_test_shap_rf = X_test_lr
X_test_shap_ffnn = X_test_s

print(f"X_test for SHAP (Random Forest) shape: {X_test_shap_rf.shape}")
print(f"X_test for SHAP (FFNN) shape: {X_test_shap_ffnn.shape}")

In [ ]:
preprocessor_rf = best_rf.named_steps['prep']
X_test_shap_rf_processed = preprocessor_rf.transform(X_test_shap_rf)

# Get feature names after preprocessing
cat_transformer = preprocessor_rf.named_transformers_['cat']
cat_feature_names = cat_transformer.get_feature_names_out(cat_features)
all_feature_names = num_features + list(cat_feature_names)

print("X_test_shap_rf_processed shape:", X_test_shap_rf_processed.shape)
print("Number of feature names:", len(all_feature_names))
# print(all_feature_names[:10]) # Display first 10 for sanity check

In [ ]:
rf_model = best_rf.named_steps['model']
explainer_rf = shap.TreeExplainer(rf_model)

# Select a subset of the processed test data (e.g., 200 samples)
np.random.seed(42) # for reproducibility
shap_subset_indices = np.random.choice(X_test_shap_rf_processed.shape[0], 200, replace=False)
X_test_shap_subset = X_test_shap_rf_processed[shap_subset_indices]

# Calculate SHAP values for the subset
shap_values_rf = explainer_rf.shap_values(X_test_shap_subset)

print(f"SHAP explainer for Random Forest created: {type(explainer_rf)}")
print(f"Shape of X_test_shap_subset: {X_test_shap_subset.shape}")
print(f"Shape of SHAP values for Random Forest (class 0, class 1): {shap_values_rf[0].shape, shap_values_rf[1].shape}")

In [ ]:
print("--- SHAP Summary Plot (Bar) for Random Forest ---")
shap.summary_plot(shap_values_rf[:, :, 1], X_test_shap_subset, feature_names=all_feature_names, plot_type="bar", show=False)
plt.title("SHAP Feature Importance (Bar) for Churn (Class 1)")
plt.tight_layout()
plt.show()

print("\n--- SHAP Summary Plot (Dot) for Random Forest ---")
shap.summary_plot(shap_values_rf[:, :, 1], X_test_shap_subset, feature_names=all_feature_names, plot_type="dot", show=False)
plt.title("SHAP Feature Impact (Dot) for Churn (Class 1)")
plt.tight_layout()
plt.show()

In [ ]:
print("--- SHAP Summary Plot (Bar) for Random Forest ---")
shap.summary_plot(shap_values_rf[:, :, 1], X_test_shap_subset, feature_names=all_feature_names, plot_type="bar", show=False)
plt.title("SHAP Feature Importance (Bar) for Churn (Class 1)")
plt.tight_layout()
plt.show()

print("\n--- SHAP Summary Plot (Dot) for Random Forest ---")
shap.summary_plot(shap_values_rf[:, :, 1], X_test_shap_subset, feature_names=all_feature_names, plot_type="dot", show=False)
plt.title("SHAP Feature Impact (Dot) for Churn (Class 1)")
plt.tight_layout()
plt.show()

In [ ]:
print("\n--- SHAP Dependence Plots for Random Forest (Top Features) ---")

# Identify top features from the summary plot (assuming these are among the most influential)
top_features_for_dependence = ['tenure', 'MonthlyCharges', 'Contract_Month-to-month']

# Get the indices of these features in the all_feature_names list
feature_indices = [all_feature_names.index(f) for f in top_features_for_dependence]

for i, feature_name in enumerate(top_features_for_dependence):
    # For interaction_index, choose another important feature or let SHAP select one
    # For simplicity, we can let SHAP automatically find the best interaction feature.
    print(f"\nDependence Plot for: {feature_name}")
    shap.dependence_plot(
        feature_indices[i], # Feature index for current feature
        shap_values_rf[:, :, 1], # SHAP values for class 1 (churn)
        X_test_shap_subset, # Data used for plotting
        feature_names=all_feature_names, # All feature names
        interaction_index="auto", # Automatically find best interaction feature
        title=f"SHAP Dependence Plot for {feature_name}",
        show=False
    )
    plt.tight_layout()
    plt.show()

In [ ]:
explainer_ffnn = shap.DeepExplainer(best_model, background_data_shap)

# Select a subset of 200 samples from the scaled test data
np.random.seed(42) # for reproducibility
shap_subset_indices_ffnn = np.random.choice(X_test_shap_ffnn.shape[0], 200, replace=False)
X_test_shap_subset_ffnn = X_test_shap_ffnn[shap_subset_indices_ffnn]

# Calculate SHAP values for the subset
# For DeepExplainer, shap_values returns a list of arrays for multi-output models,
# or a single array for single-output models. Here, it's a binary classification
# with a single output (sigmoid), so it returns a single array for the output logit.
# However, to maintain consistency with TreeExplainer's multi-class output format,
# we might need to adjust how we pass/handle the output for plotting if it's a single array.
# Let's calculate and inspect its shape.
shap_values_ffnn = explainer_ffnn.shap_values(X_test_shap_subset_ffnn)

print(f"SHAP explainer for Raw FFNN created: {type(explainer_ffnn)}")
print(f"Shape of X_test_shap_subset_ffnn: {X_test_shap_subset_ffnn.shape}")
print(f"Shape of SHAP values for Raw FFNN: {np.array(shap_values_ffnn).shape}")


In [ ]:
import matplotlib.pyplot as plt

# The FFNN was trained on X_encoded, so the feature names should correspond to X_encoded.columns
# X_encoded is already created in the notebook, so I'll reuse its columns for feature names.
feature_names_ffnn = X_encoded.columns.tolist()

print("--- SHAP Summary Plot (Bar) for Raw FFNN (Churn Class) ---")
shap.summary_plot(shap_values_ffnn.squeeze(), X_test_shap_subset_ffnn, feature_names=feature_names_ffnn, plot_type="bar", show=False)
plt.title("SHAP Feature Importance (Bar) for Churn (Class 1) - FFNN")
plt.tight_layout()
plt.show()

print("\n--- SHAP Summary Plot (Dot) for Raw FFNN (Churn Class) ---")
shap.summary_plot(shap_values_ffnn.squeeze(), X_test_shap_subset_ffnn, feature_names=feature_names_ffnn, plot_type="dot", show=False)
plt.title("SHAP Feature Impact (Dot) for Churn (Class 1) - FFNN")
plt.tight_layout()
plt.show()

In [ ]:
print("\n--- SHAP Dependence Plots for Raw FFNN (Top Features) ---")

# Identify top features from the summary plot (or use common influential ones)
top_features_for_dependence_ffnn = ['tenure', 'MonthlyCharges', 'TotalCharges']

# The FFNN was trained on X_encoded, so the feature names should correspond to X_encoded.columns
feature_names_ffnn = X_encoded.columns.tolist()

for i, feature_name in enumerate(top_features_for_dependence_ffnn):
    # For DeepExplainer, shap_values_ffnn is already for a single output (class 1 churn)
    # The feature index needs to be obtained from the feature_names_ffnn list.
    feature_idx = feature_names_ffnn.index(feature_name)

    print(f"\nDependence Plot for: {feature_name}")
    shap.dependence_plot(
        feature_idx, # Feature index for current feature
        shap_values_ffnn.squeeze(), # SHAP values for class 1 (churn), squeezed to (n_samples, n_features)
        X_test_shap_subset_ffnn, # Data used for plotting
        feature_names=feature_names_ffnn, # All feature names
        interaction_index="auto", # Automatically find best interaction feature
        title=f"SHAP Dependence Plot for {feature_name} - FFNN",
        show=False
    )
    plt.tight_layout()
    plt.show()